# Análise: Preço de Alimentos e Obesidade nos EUA
**Datasets:** CDC BRFSS + USDA FMAP | **Período:** 2012–2018

Pipeline:
1. Carregar dados brutos com Pandas
2. Toda a limpeza, filtragem e cruzamento via **DuckDB (SQL)**
3. Análise exploratória com SQL

## Etapa 1 — Importações e carregamento dos dados

In [21]:
import pandas as pd
import duckdb

df_cdc  = pd.read_csv("data/cdc_dados.csv", low_memory=False)
df_usda = pd.read_csv("data/FMAP-Data.csv")

print(f"CDC  → {df_cdc.shape[0]:,} linhas x {df_cdc.shape[1]} colunas")
print(f"USDA → {df_usda.shape[0]:,} linhas x {df_usda.shape[1]} colunas")

CDC  → 110,880 linhas x 33 colunas
USDA → 1,020,600 linhas x 6 colunas


## Etapa 2 — Registrar tabelas no DuckDB

O DuckDB permite consultar DataFrames Pandas diretamente com SQL, sem precisar de banco de dados externo.

In [22]:
con = duckdb.connect()

con.register("cdc_raw", df_cdc)
con.register("usda_raw", df_usda)

print("Tabelas disponíveis no DuckDB:")
display(con.execute("SHOW TABLES").df())

Tabelas disponíveis no DuckDB:


,name
0,cdc_raw
1,usda_raw


## Etapa 3 — Exploração do CDC via SQL

In [23]:
print("Classes disponíveis no CDC:")
display(con.execute("""
    SELECT Class, COUNT(*) AS total
    FROM cdc_raw
    GROUP BY Class
""").df())

print("\nAnos disponíveis:")
display(con.execute("""
    SELECT DISTINCT YearStart AS Year
    FROM cdc_raw
    ORDER BY Year
""").df())

print("\nFaixas de renda:")
display(con.execute("""
    SELECT DISTINCT Stratification1 AS Income_Group
    FROM cdc_raw
    WHERE StratificationCategory1 = 'Income'
    ORDER BY Income_Group
""").df())

Classes disponíveis no CDC:


,Class,total
0,Obesity / Weight Status,43120
1,Fruits and Vegetables,9240
2,Physical Activity,58520



Anos disponíveis:


,Year
0,2011
1,2012
2,2013
3,2014
4,2015
5,2016
6,2017
7,2018
8,2019
9,2020



Faixas de renda:


,Income_Group
0,"$15,000 - $24,999"
1,"$25,000 - $34,999"
2,"$35,000 - $49,999"
3,"$50,000 - $74,999"
4,"$75,000 or greater"
5,Data not reported
6,"Less than $15,000"


## Etapa 4 — Limpeza e filtragem do CDC via SQL

Filtramos:
- Apenas `Obesity / Weight Status`
- Estratificação por renda, excluindo `Data not reported`
- Período 2012–2018
- Apenas os 50 estados + DC

In [24]:
cdc_filtrado = con.execute("""
    SELECT
        YearStart       AS Year,
        LocationAbbr    AS State,
        Stratification1 AS Income_Group,
        Data_Value      AS Obesity_Rate
    FROM cdc_raw
    WHERE
        Class                   = 'Obesity / Weight Status'
        AND StratificationCategory1 = 'Income'
        AND Stratification1     != 'Data not reported'
        AND YearStart           BETWEEN 2012 AND 2018
        AND Data_Value          IS NOT NULL
        AND LocationAbbr NOT IN ('US', 'PR', 'GU', 'VI')
    ORDER BY Year, State, Income_Group
""").df()

con.register("cdc_filtrado", cdc_filtrado)

print(f"Linhas antes: {len(df_cdc):,}")
print(f"Linhas depois: {len(cdc_filtrado):,}")
display(cdc_filtrado.head(10))

Linhas antes: 110,880
Linhas depois: 4,284


,Year,State,Income_Group,Obesity_Rate
0,2012,AK,"$15,000 - $24,999",24.2
1,2012,AK,"$15,000 - $24,999",40.3
2,2012,AK,"$25,000 - $34,999",24.0
3,2012,AK,"$25,000 - $34,999",38.0
4,2012,AK,"$35,000 - $49,999",27.3
5,2012,AK,"$35,000 - $49,999",41.6
6,2012,AK,"$50,000 - $74,999",37.8
7,2012,AK,"$50,000 - $74,999",28.4
8,2012,AK,"$75,000 or greater",40.8
9,2012,AK,"$75,000 or greater",26.4


## Etapa 5 — Exploração do USDA via SQL

In [25]:
print("Atributos disponíveis no USDA:")
display(con.execute("""
    SELECT DISTINCT Attribute, COUNT(*) AS total
    FROM usda_raw
    GROUP BY Attribute
    ORDER BY total DESC
""").df())

print("\nRegiões disponíveis:")
display(con.execute("""
    SELECT DISTINCT Metroregion_code
    FROM usda_raw
    ORDER BY Metroregion_code
""").df())

Atributos disponíveis no USDA:


,Attribute,total
0,Purchase_grams_wtd,113400
1,Unit_value_mean_wtd,113400
2,Price_index_GEKS,113400
3,Number_stores,113400
4,Unit_value_mean_unwtd,113400
5,Purchase_dollars_unwtd,113400
6,Unit_value_se_wtd,113400
7,Purchase_dollars_wtd,113400
8,Purchase_grams_unwtd,113400



Regiões disponíveis:


,Metroregion_code
0,0
1,1
2,2
3,3
4,4
5,12060
6,14460
7,16980
8,19100
9,19820


## Etapa 6 — Classificação e filtragem do USDA via SQL

Usamos `CASE WHEN` para classificar cada código como `healthy` ou `processed`.

In [26]:
usda_filtrado = con.execute("""
    SELECT
        Year,
        Month,
        Metroregion_code,
        EFPG_code,
        Value,
        CASE
            WHEN EFPG_code IN (
                10000, 10025, 10050, 10075,
                20000, 21500, 21525, 21550,
                23000, 24500, 24525, 24550,
                26000, 26525, 26550,
                27500, 27550,
                29000, 29025, 29050,
                30000, 30025, 30050, 30090,
                50000, 50050,
                53000, 53050,
                54500, 54550,
                57500, 59000
            ) THEN 'healthy'
            WHEN EFPG_code IN (
                15000, 15025, 15050, 15075,
                46050, 50075, 56000,
                60000, 62500, 65000, 67500,
                72000, 72020, 72040, 72050,
                73000, 73010, 73020, 73030,
                73040, 73050, 73060, 75050,
                70000, 70050
            ) THEN 'processed'
        END AS Food_Type
    FROM usda_raw
    WHERE
        Attribute        = 'Unit_value_mean_wtd'
        AND Metroregion_code IN (1, 2, 3, 4)
        AND Food_Type    IS NOT NULL
    ORDER BY Year, Metroregion_code, EFPG_code
""").df()

con.register("usda_filtrado", usda_filtrado)

print(f"Linhas antes: {len(df_usda):,}")
print(f"Linhas depois: {len(usda_filtrado):,}")
print("\nDistribuição por tipo:")
display(con.execute("""
    SELECT Food_Type, COUNT(*) AS total
    FROM usda_filtrado
    GROUP BY Food_Type
""").df())
display(usda_filtrado.head(10))

Linhas antes: 1,020,600
Linhas depois: 19,152

Distribuição por tipo:


,Food_Type,total
0,processed,8400
1,healthy,10752


,Year,Month,Metroregion_code,EFPG_code,Value,Food_Type
0,2012,1,1,10000,0.551337,healthy
1,2012,2,1,10000,0.552195,healthy
2,2012,3,1,10000,0.549603,healthy
3,2012,4,1,10000,0.548377,healthy
4,2012,5,1,10000,0.545741,healthy
5,2012,6,1,10000,0.544072,healthy
6,2012,7,1,10000,0.546840,healthy
7,2012,8,1,10000,0.542625,healthy
8,2012,9,1,10000,0.547550,healthy
9,2012,10,1,10000,0.546840,healthy


## Etapa 7 — Agregação anual do USDA via SQL

O USDA é mensal e o CDC é anual. Calculamos a média anual por região e tipo.

In [27]:
usda_pivot = con.execute("""
    SELECT
        Year,
        Metroregion_code,
        AVG(CASE WHEN Food_Type = 'healthy'   THEN Value END) AS Price_Healthy,
        AVG(CASE WHEN Food_Type = 'processed' THEN Value END) AS Price_Processed,
        AVG(CASE WHEN Food_Type = 'healthy'   THEN Value END) /
        AVG(CASE WHEN Food_Type = 'processed' THEN Value END) AS Price_Ratio
    FROM usda_filtrado
    GROUP BY Year, Metroregion_code
    ORDER BY Year, Metroregion_code
""").df()

con.register("usda_pivot", usda_pivot)

print(f"Shape: {usda_pivot.shape}")
display(usda_pivot)

Shape: (28, 5)


,Year,Metroregion_code,Price_Healthy,Price_Processed,Price_Ratio
0,2012,1,0.623113,0.556894,1.118908
1,2012,2,0.563835,0.507985,1.109946
2,2012,3,0.573587,0.517477,1.108431
3,2012,4,0.575772,0.555885,1.035776
4,2013,1,0.635479,0.588926,1.079047
5,2013,2,0.575229,0.515135,1.116656
6,2013,3,0.585068,0.526139,1.112002
7,2013,4,0.594675,0.567844,1.047249
8,2014,1,0.644572,0.586464,1.099081
9,2014,2,0.591520,0.523685,1.129534


## Etapa 8 — Tabela de mapeamento estado → região

In [28]:
mapeamento = [
    *[(s, 1) for s in ["CT","ME","MA","NH","NJ","NY","PA","RI","VT"]],
    *[(s, 2) for s in ["IL","IN","IA","KS","MI","MN","MO","NE","ND","OH","SD","WI"]],
    *[(s, 3) for s in ["AL","AR","DE","FL","GA","KY","LA","MD","MS","NC","OK","SC","TN","TX","VA","WV","DC"]],
    *[(s, 4) for s in ["AK","AZ","CA","CO","HI","ID","MT","NV","NM","OR","UT","WA","WY"]],
]

df_mapeamento = pd.DataFrame(mapeamento, columns=["State", "Metroregion_code"])
con.register("state_region", df_mapeamento)

print(f"Total de estados mapeados: {len(df_mapeamento)}")
display(df_mapeamento.head(10))

Total de estados mapeados: 51


,State,Metroregion_code
0,CT,1
1,ME,1
2,MA,1
3,NH,1
4,NJ,1
5,NY,1
6,PA,1
7,RI,1
8,VT,1
9,IL,2


## Etapa 9 — Tabela analítica final via SQL

JOIN entre CDC, mapeamento de regiões e USDA em uma única query.

In [29]:
df_final = con.execute("""
    SELECT
        c.Year,
        sr.Metroregion_code,
        c.Income_Group,
        ROUND(AVG(c.Obesity_Rate),    4) AS Obesity_Rate,
        ROUND(AVG(u.Price_Healthy),   6) AS Price_Healthy,
        ROUND(AVG(u.Price_Processed), 6) AS Price_Processed,
        ROUND(AVG(u.Price_Ratio),     6) AS Price_Ratio
    FROM      cdc_filtrado  c
    JOIN      state_region  sr ON c.State = sr.State
    JOIN      usda_pivot    u  ON c.Year  = u.Year
                               AND sr.Metroregion_code = u.Metroregion_code
    GROUP BY  c.Year, sr.Metroregion_code, c.Income_Group
    ORDER BY  c.Year, sr.Metroregion_code, c.Income_Group
""").df()

con.register("tabela_analitica", df_final)

print(f"Tabela analítica final: {df_final.shape}")
print(f"Colunas: {df_final.columns.tolist()}")
print(f"Nulos: {df_final.isnull().sum().sum()}")
display(df_final.head(12))

df_final.to_csv("data/tabela_analitica.csv", index=False)
print("\nSalvo em: data/tabela_analitica.csv")

Tabela analítica final: (168, 7)
Colunas: ['Year', 'Metroregion_code', 'Income_Group', 'Obesity_Rate', 'Price_Healthy', 'Price_Processed', 'Price_Ratio']
Nulos: 0


,Year,Metroregion_code,Income_Group,Obesity_Rate,Price_Healthy,Price_Processed,Price_Ratio
0,2012,1,"$15,000 - $24,999",32.2000,0.623113,0.556894,1.118908
1,2012,1,"$25,000 - $34,999",32.0000,0.623113,0.556894,1.118908
2,2012,1,"$35,000 - $49,999",32.5444,0.623113,0.556894,1.118908
3,2012,1,"$50,000 - $74,999",32.5167,0.623113,0.556894,1.118908
4,2012,1,"$75,000 or greater",30.2944,0.623113,0.556894,1.118908
5,2012,1,"Less than $15,000",31.5667,0.623113,0.556894,1.118908
6,2012,2,"$15,000 - $24,999",33.1500,0.563835,0.507985,1.109946
7,2012,2,"$25,000 - $34,999",33.5917,0.563835,0.507985,1.109946
8,2012,2,"$35,000 - $49,999",34.2208,0.563835,0.507985,1.109946
9,2012,2,"$50,000 - $74,999",34.1208,0.563835,0.507985,1.109946



Salvo em: data/tabela_analitica.csv


## Etapa 10 — Análise exploratória via SQL

Consultas que testam as hipóteses H1, H2 e H3 do artigo.

In [30]:
print("Q1 — Evolução do preço saudável vs. processado por ano (H1)")
display(con.execute("""
    SELECT
        Year,
        ROUND(AVG(Price_Healthy),   4) AS avg_price_healthy,
        ROUND(AVG(Price_Processed), 4) AS avg_price_processed,
        ROUND(AVG(Price_Ratio),     4) AS avg_price_ratio,
        ROUND(AVG(Obesity_Rate),    2) AS avg_obesity
    FROM tabela_analitica
    GROUP BY Year
    ORDER BY Year
""").df())

Q1 — Evolução do preço saudável vs. processado por ano (H1)


,Year,avg_price_healthy,avg_price_processed,avg_price_ratio,avg_obesity
0,2012,0.5841,0.5346,1.0933,32.35
1,2013,0.5976,0.5495,1.0887,32.58
2,2014,0.6099,0.5509,1.1081,32.84
3,2015,0.6302,0.5655,1.1148,33.04
4,2016,0.6302,0.5689,1.1078,33.05
5,2017,0.6321,0.5726,1.1047,33.43
6,2018,0.6483,0.5837,1.1117,33.54


In [31]:
print("Q2 — Obesidade média por faixa de renda (H3)")
display(con.execute("""
    SELECT
        Income_Group,
        ROUND(AVG(Obesity_Rate), 2) AS avg_obesity,
        ROUND(MIN(Obesity_Rate), 2) AS min_obesity,
        ROUND(MAX(Obesity_Rate), 2) AS max_obesity,
        COUNT(*) AS n
    FROM tabela_analitica
    GROUP BY Income_Group
    ORDER BY avg_obesity DESC
""").df())

Q2 — Obesidade média por faixa de renda (H3)


,Income_Group,avg_obesity,min_obesity,max_obesity,n
0,"$50,000 - $74,999",33.82,32.13,35.72,28
1,"$35,000 - $49,999",33.71,31.39,35.74,28
2,"$25,000 - $34,999",33.13,31.22,34.77,28
3,"$15,000 - $24,999",32.96,30.97,34.93,28
4,"Less than $15,000",32.13,29.72,34.13,28
5,"$75,000 or greater",32.11,30.06,34.42,28


In [32]:
print("Q3 — Correlação entre Price_Ratio e Obesity_Rate por região")
display(con.execute("""
    SELECT
        Metroregion_code,
        ROUND(CORR(Price_Ratio,   Obesity_Rate), 4) AS corr_ratio_obesity,
        ROUND(CORR(Price_Healthy, Obesity_Rate), 4) AS corr_healthy_obesity,
        COUNT(*) AS n
    FROM tabela_analitica
    GROUP BY Metroregion_code
    ORDER BY Metroregion_code
""").df())

Q3 — Correlação entre Price_Ratio e Obesity_Rate por região


,Metroregion_code,corr_ratio_obesity,corr_healthy_obesity,n
0,1,-0.1054,0.3282,42
1,2,0.3570,0.3948,42
2,3,-0.0472,0.5175,42
3,4,0.4784,0.4830,42


In [33]:
print("Q4 — Renda baixa vs. renda alta por região e ano (H3)")
display(con.execute("""
    SELECT
        Year,
        Metroregion_code,
        ROUND(AVG(CASE WHEN Income_Group = 'Less than $15,000'  THEN Obesity_Rate END), 2) AS obesity_renda_baixa,
        ROUND(AVG(CASE WHEN Income_Group = '$75,000 or greater' THEN Obesity_Rate END), 2) AS obesity_renda_alta,
        ROUND(
            AVG(CASE WHEN Income_Group = 'Less than $15,000'  THEN Obesity_Rate END) -
            AVG(CASE WHEN Income_Group = '$75,000 or greater' THEN Obesity_Rate END)
        , 2) AS diferenca
    FROM tabela_analitica
    GROUP BY Year, Metroregion_code
    ORDER BY Year, Metroregion_code
""").df())

Q4 — Renda baixa vs. renda alta por região e ano (H3)


,Year,Metroregion_code,obesity_renda_baixa,obesity_renda_alta,diferenca
0,2012,1,31.57,30.29,1.27
1,2012,2,30.94,32.43,-1.49
2,2012,3,32.70,32.33,0.37
3,2012,4,29.97,30.43,-0.47
4,2013,1,31.16,30.32,0.83
5,2013,2,31.37,32.85,-1.48
6,2013,3,33.81,32.18,1.62
7,2013,4,29.72,30.54,-0.82
8,2014,1,31.24,30.06,1.18
9,2014,2,32.36,33.14,-0.78


In [34]:
print("Q5 — Estatísticas descritivas finais")
display(con.execute("""
    SELECT
        COUNT(*)                         AS total_linhas,
        COUNT(DISTINCT Year)             AS anos,
        COUNT(DISTINCT Metroregion_code) AS regioes,
        COUNT(DISTINCT Income_Group)     AS faixas_renda,
        ROUND(MIN(Obesity_Rate), 2)      AS min_obesity,
        ROUND(MAX(Obesity_Rate), 2)      AS max_obesity,
        ROUND(AVG(Obesity_Rate), 2)      AS avg_obesity,
        ROUND(MIN(Price_Ratio),  4)      AS min_ratio,
        ROUND(MAX(Price_Ratio),  4)      AS max_ratio,
        ROUND(AVG(Price_Ratio),  4)      AS avg_ratio
    FROM tabela_analitica
""").df())

Q5 — Estatísticas descritivas finais


,total_linhas,anos,regioes,faixas_renda,min_obesity,max_obesity,avg_obesity,min_ratio,max_ratio,avg_ratio
0,168,7,4,6,29.72,35.74,32.98,1.0358,1.1393,1.1042


In [35]:
print("Q1 — Evolução do preço saudável vs. processado por ano (H1)")
display(con.execute("""
    SELECT
        Year,
        ROUND(AVG(Price_Healthy),   4) AS avg_price_healthy,
        ROUND(AVG(Price_Processed), 4) AS avg_price_processed,
        ROUND(AVG(Price_Ratio),     4) AS avg_price_ratio,
        ROUND(AVG(Obesity_Rate),    2) AS avg_obesity
    FROM tabela_analitica
    GROUP BY Year
    ORDER BY Year
""").df())

Q1 — Evolução do preço saudável vs. processado por ano (H1)


,Year,avg_price_healthy,avg_price_processed,avg_price_ratio,avg_obesity
0,2012,0.5841,0.5346,1.0933,32.35
1,2013,0.5976,0.5495,1.0887,32.58
2,2014,0.6099,0.5509,1.1081,32.84
3,2015,0.6302,0.5655,1.1148,33.04
4,2016,0.6302,0.5689,1.1078,33.05
5,2017,0.6321,0.5726,1.1047,33.43
6,2018,0.6483,0.5837,1.1117,33.54


In [36]:
print("Q2 — Obesidade média por faixa de renda (H3)")
display(con.execute("""
    SELECT
        Income_Group,
        ROUND(AVG(Obesity_Rate), 2) AS avg_obesity,
        ROUND(MIN(Obesity_Rate), 2) AS min_obesity,
        ROUND(MAX(Obesity_Rate), 2) AS max_obesity,
        COUNT(*) AS n
    FROM tabela_analitica
    GROUP BY Income_Group
    ORDER BY avg_obesity DESC
""").df())

Q2 — Obesidade média por faixa de renda (H3)


,Income_Group,avg_obesity,min_obesity,max_obesity,n
0,"$50,000 - $74,999",33.82,32.13,35.72,28
1,"$35,000 - $49,999",33.71,31.39,35.74,28
2,"$25,000 - $34,999",33.13,31.22,34.77,28
3,"$15,000 - $24,999",32.96,30.97,34.93,28
4,"Less than $15,000",32.13,29.72,34.13,28
5,"$75,000 or greater",32.11,30.06,34.42,28


In [37]:
print("Q3 — Correlação entre Price_Ratio e Obesity_Rate por região")
display(con.execute("""
    SELECT
        Metroregion_code,
        ROUND(CORR(Price_Ratio,   Obesity_Rate), 4) AS corr_ratio_obesity,
        ROUND(CORR(Price_Healthy, Obesity_Rate), 4) AS corr_healthy_obesity,
        COUNT(*) AS n
    FROM tabela_analitica
    GROUP BY Metroregion_code
    ORDER BY Metroregion_code
""").df())

Q3 — Correlação entre Price_Ratio e Obesity_Rate por região


,Metroregion_code,corr_ratio_obesity,corr_healthy_obesity,n
0,1,-0.1054,0.3282,42
1,2,0.3570,0.3948,42
2,3,-0.0472,0.5175,42
3,4,0.4784,0.4830,42


In [38]:
print("Q4 — Renda baixa vs. renda alta por região e ano (H3)")
display(con.execute("""
    SELECT
        Year,
        Metroregion_code,
        ROUND(AVG(CASE WHEN Income_Group = 'Less than $15,000'
              THEN Obesity_Rate END), 2) AS obesity_renda_baixa,
        ROUND(AVG(CASE WHEN Income_Group = '$75,000 or greater'
              THEN Obesity_Rate END), 2) AS obesity_renda_alta,
        ROUND(
            AVG(CASE WHEN Income_Group = 'Less than $15,000'
                THEN Obesity_Rate END) -
            AVG(CASE WHEN Income_Group = '$75,000 or greater'
                THEN Obesity_Rate END)
        , 2) AS diferenca
    FROM tabela_analitica
    GROUP BY Year, Metroregion_code
    ORDER BY Year, Metroregion_code
""").df())

Q4 — Renda baixa vs. renda alta por região e ano (H3)


,Year,Metroregion_code,obesity_renda_baixa,obesity_renda_alta,diferenca
0,2012,1,31.57,30.29,1.27
1,2012,2,30.94,32.43,-1.49
2,2012,3,32.70,32.33,0.37
3,2012,4,29.97,30.43,-0.47
4,2013,1,31.16,30.32,0.83
5,2013,2,31.37,32.85,-1.48
6,2013,3,33.81,32.18,1.62
7,2013,4,29.72,30.54,-0.82
8,2014,1,31.24,30.06,1.18
9,2014,2,32.36,33.14,-0.78
